# <center>VAI Academy</center>
# <center> Aquisição e Preparação de Dados - Parte 2 </center>
___

# <center> Objetivos de aprendizado </center>
- Familiarizar-se com a obtenção de dados de arquivos flat files e provenientes do Excel
- Familiarizar-se com a obtenção de dados de bancos de dados relacionais através de consultas SQL
___

## Conteúdo
1. [Introdução](#intro)
2. [Aquisição de Dados](#dataacq)

<a id="dsprecap"></a>
## Recapitulando: Data Science Pipeline (DSP)
1. Entendimento do Problema / Definição de Escopo
2. Definição de Objetivo e Configuração de Métricas
3. Determinação dos Dados Necessários
4. Aquisição de Dados
5. Limpeza de Dados & Manipulação
6. Análise Exploratória de Dados (E.D.A.)
7. Engenharia de Atributos
8. Construção e Avaliação do Modelo
9. Interpretação de Resultados & Relatório / Data Storytelling
10. Implantação
11. Monitoramento e Manutenção

**Esta aula contemplará os conceitos dos passos 4 e 5.**

In [ ]:
# Instalando alguns pacotes necessários ao longo do notebook
# ATENÇÃO:  Após instalar as bibliotecas reinicie o ambiente de execução para garantir que está usando a versão correta
!pip install xlrd==2.0.1
!pip install pandas==2.1.1

Dando seguimento aos processos de aquisição de dados......

### 2.6. Parquet

Parquet é um formato para armazenamento de dados de código aberto disponível em todo ecossistema [Hadoop](https://en.wikipedia.org/wiki/Apache_Hadoop). Diferente dos formatos tradicionais que armazenam os dados orientados a linhas, o Parquet armazena os dados de forma colunar, observe na imagem abaixo:

<img src=https://help.sap.com/doc/PRODUCTION/6b94445c94ae495c83a19646e7c3fd56/2.0.00/en-US/loio7156b2d7b35b4b2abda3c00057ddc210_LowRes.png alt="drawing" width="400"/>

Esse tipo de armazenamento possui algumas vantagens:
- Os dados na forma colunar do Parquet podem ser comprimidos de forma mais eficiente, ocupando menos espaço
- No formato colunar, ao realizar uma consulta de dados (que você vai aprender daqui a pouco), é possível pular os dados não relevantes mais rapidamente (pois no Parquet o armazenamento ocorre por coluna, permitindo maior eficiência na hora de pular os dados desnecessários)

Em um [comparativo realizado pela empresa Databricks](https://databricks.com/glossary/what-is-parquet), obteve-se os seguintes resultados comparando o CSV com o Parquet:


| Formato |	Espaço utilizado |	Tempo de execução |
|:-------:|:-----:|:-----:|
|CSV|1 TB|236 seg|
|Parquet|130 GB|6.78 seg|

O uso de Parquet reduziu o espaço de armazenamento em 87% e executou 34x mais rápido em determinadas operações.
Viu como o uso do Parquet pode ser interessante? Para ler um arquivo neste formato é bem simples, usamos o ```.read_parquet()```, similar ao ```.read_csv()``` que vimos anteriormente:

In [ ]:
import pandas as pd

df_parquet = pd.read_parquet('dados/sample.parquet')
df_parquet

### 2.7. Base de dados

Agora vamos lhe ensinar como conectar Python a um banco de dados relacional. Primeiro de tudo, o que é um banco de dados relacional?

Um banco de dados (DB) é uma coleção eletrônica organizada de dados. Formalmente, é um [banco de dados relacional](https://cloud.google.com/learn/what-is-a-relational-database?hl=pt-br) se seguir um modelo relacional de dados. Para simplificar, as bases de dados relacionais são as que têm:

- tabelas que representam entidades (como um dataframe);
- linhas, ou registros, representando instâncias de uma entidade;
- colunas que representam atributos ou características de uma instância;
- chaves primárias para cada tabela, ou seja, colunas com entradas únicas para cada linha;
- relações entre tabelas;

Os sistemas que gerenciam esses bancos de dados são chamados de *Relational Database Management Systems*, ou RDBMS. Você pode conhecer os mais famosos deles, tais como PostgreSQL, MySQL ou SQLite. Praticamente todos os RDBMS seguem a Structured Query Language (SQL), uma linguagem específica para trabalhar com este tipo de banco de dados.

Primeiro, vamos mostrar como se conectar facilmente a um banco de dados e executar uma simples consulta SQL. Em seguida, vamos aprofundar um pouco em SQL para torná-lo capaz de construir suas próprias consultas para adquirir dados do banco de dados.

#### 2.7.1. Conexão

A primeira coisa a fazer para usar um banco de dados em Python é conectar-se com ele. O problema com isso é que a forma de fazer isso depende de qual é o RDBMS do seu database.

Para nos ajudar com isso e dar uma interface comum para trabalhar com bancos de dados relacionais, vem o pacote [```sqlalchemy```](https://www.sqlalchemy.org/). Este pacote nos permite criar, com uma única função, uma **engine** para qualquer RDBMS. Assim, podemos utilizar esse motor para conectar ao banco de dados para acessar os dados nele contidos.

A conexão funciona exatamente como um arquivo, como vimos quando aprendemos ```pickle```. O criamos, fazemos o que quisermos com ele, então devemos fechá-lo.

Chega de conversa, vamos vê-lo funcionando.

In [ ]:
# primeiro importamos a função create_engine de sqlalchemy
import sqlalchemy

# entao criamos a engine passando a string de conexão
engine_str = 'sqlite:///dados/chinook.sqlite'
engine = sqlalchemy.create_engine(engine_str)

# já temos a engine! Vamos ver as tabelas em um DB
table_names = sqlalchemy.inspect(engine).get_table_names()
print(table_names)

Não é difícil, certo?

Estamos usando um banco de dados muito simples, o SQLite. É tão simples, que armazena todas as tabelas em um único arquivo .sqlite e não precisa de usuário ou senha. Por causa disso, nosso ```engine_str``` é muito simples, apenas definindo o tipo de DB com **'sqlite'** antes do **'://'** e o caminho para o arquivo depois dele **'/dados/chinook.sqlite'**.

Formato da string: ```sqlite://<caminho do arquivo>```, note que o caminho deve começar com ```/nome da pasta/...```


Se você precisar se conectar a outro banco de dados que requeira mais informações, dê uma olhada na [documentação](https://docs.sqlalchemy.org/en/latest/core/engines.html) para ```create_engine```.

Agora que temos o *engine*, podemos nos conectar, executar uma consulta e desconectar do banco de dados.

In [ ]:
# cria a conexão
con = engine.connect()

# escreve uma consulta/query simples que seleciona todas as linhas e colunas da tabela Customer - vamos explicar mais sobre queries na próxima seção
query = sqlalchemy.text("SELECT * FROM Customer")

# executa a query com a conexão
rs = con.execute(query)

# fetch/get todos os resultados
results = rs.fetchall()

# cria o dataframe
df = pd.DataFrame(results)
df.info()
print('\n')

# pega as colunas da conexão
df.columns = rs.keys()
df.info()

# fecha a conexão
con.close()

É isso aí! Crie a *engine*, conecte, execute a consulta, busque todos os resultados e nomes de colunas, feche a conexão.

E se nos esquecermos de fechar? Nunca devemos fazer isso. Assim, como com os arquivos, podemos utilizar o "with". Veja abaixo.

In [ ]:
with engine.connect() as con:                  # conecta dentro do with
    query = sqlalchemy.text("SELECT * FROM Album")
    rs = con.execute(query)    # executa a query
    df = pd.DataFrame(rs.fetchmany(size=5))    # pega os resultados - aqui usamos fetchmany para pegar o numero específico de linhas
    df.columns = rs.keys()                     # pega o nome das colunas

df.info()

Um pouco mais fácil com ```with``` e temos certeza de que a conexão está sendo fechada!

Uma última maneira de fazer isso é utilizando a função pandas ```read_sql_query```, onde você só precisa passar o engine e a query e ela retorna um dataframe diretamente, dê uma olhada.

In [ ]:
# utiliza o pandas read_sql_query para ler o resultado da query como dataframe
df = pd.read_sql_query("SELECT * FROM Artist", engine)
df.head()

Soluções de uma linha (completas) são sempre ótimas, não são?

É isso para a parte de conexão do banco de dados. Agora vamos aprender mais sobre como consultar com SQL.

### 2.8. SQL

SQL significa Structured Query Language e é uma linguagem padrão para armazenamento, manipulação e recuperação de dados em bancos de dados. Assim, aprender SQL permitirá obter os dados que você deseja de muitos [RDBMS](https://en.wikipedia.org/wiki/Relational_database_management_system), tais como MS SQL Server, IBM DB2, Oracle, MySQL e Microsoft Access, facilitando a análise de dados.

> **Nota***: A maioria dos programas de banco de dados SQL também tem suas próprias extensões proprietárias além do padrão SQL, mas todos eles suportam pelo menos os principais comandos que estaremos cobrindo nesta aula de maneira semelhante

Mas o que é um banco de dados? Um banco de dados contém, na maioria das vezes, uma ou mais tabelas. Cada tabela é identificada por um nome (por exemplo, "Customer" ou "Playlist" abaixo) e as tabelas contêm registros (linhas) com dados.

Neste tutorial usaremos o conhecido [Chinook](https://github.com/joellegg/14-chinook-ERD), exemplo de banco de dados (incluído no MS Access e MS SQL Server), cujo [Entity Relationship Diagram (ERD)](https://en.wikipedia.org/wiki/Entity%E2%80%93relationship_model) pode ser visto abaixo:


<img src=https://raw.githubusercontent.com/joellegg/14-chinook-ERD/master/Chinook-ERD.png alt="drawing" width="700"/>


#### 2.8.1. SELECT

A declaração SELECT é utilizada para selecionar dados de um banco de dados. Sua sintaxe é bastante simples:

```mysql
SELECT coluna1, coluna2, ...
FROM nome_da_tabela;
```

Aqui, ```coluna1```, ```coluna2```, ... são os nomes das colunas da tabela "nome_da_tabela", da qual você deseja selecionar os dados. Se você quiser selecionar todos os campos disponíveis na tabela, utilize a seguinte sintaxe:

```mysql
SELECT *
FROM nome_da_tabela;
```

#### Exemplo 2.1

Vamos executar uma select query em nosso banco de dados ```chinook``` para obter o nome de todos os clientes e a cidade onde eles moram. Para isso, podemos olhar o banco de dados ERD ou executar este comando para ver as tabelas disponíveis no banco de dados:

In [ ]:
table_names = sqlalchemy.inspect(engine).get_table_names() #Pega o nome de todas as tabelas no database
print(table_names)

Temos uma tabela ```Customer```, então precisamos selecionar os dados dessa tabela para recuperar as informações desejadas.

In [ ]:
# Querying com pandas
df = pd.read_sql_query("SELECT FirstName, LastName, City FROM Customer", engine)
print(df.head())

> **Tip**: Tenha em mente que as palavras-chave SQL são **NÃO** sensíveis a maiúsculas/minúsculas: "select" é o mesmo que "SELECT". Para estabelecer um padrão, escreveremos todas as palavras-chave SQL em maiúsculas.

#### 2.8.2. SELECT DISTINCT

A declaração SELECT DISTINCT é utilizada para retornar somente valores distintos (diferentes).

Dentro de uma tabela, uma coluna frequentemente contém muitos valores duplicados e às vezes você só quer listar os diferentes valores (distintos).

#### Exemplo 2.2

Vamos supor que queremos descobrir de quais países nossos ```customers``` são. Utilizando uma declaração SELECT normal, nós obteríamos:

In [ ]:
# seleciona todos os países da tabela de clientes
str_query = "SELECT Country FROM Customer"
df = pd.read_sql_query(str_query, engine)
print(df)

Agora usando o modificador ```DISTINCT```, nós temos:

In [ ]:
# seleciona todos os países distintos DISTINCT da tabela de clientes
str_query = "SELECT DISTINCT Country FROM Customer"
df = pd.read_sql_query(str_query, engine)
print(df)


#### 2.8.3. WHERE

A cláusula WHERE é utilizada para filtrar registros, ou seja, para extrair somente aqueles registros que preenchem uma ```condição``` especificada.

```mysql
SELECT coluna1, coluna2, ...
FROM nome_da_tabela
WHERE condição;
```
Os seguintes operadores podem ser utilizados na cláusula [```WHERE```](https://www.w3schools.com/sql/sql_where.asp):

- ```=```: Igual

- ```>```: Maior do que

- ```<```: Menor do de

- ```>=```: Maior ou igual

- ```<=```: Menor ou igual

- ```<>```: Não é igual. **Nota**: Em algumas versões do SQL, este operador pode ser escrito como !=

- ```BETWEEN```: Entre um determinado intervalo

- ```LIKE```: Busca por um padrão

- ```IN```: Para especificar múltiplos valores possíveis para uma coluna

**Nota**: SQL requer aspas simples em torno de valores de texto (a maioria dos sistemas de banco de dados também permitirá aspas duplas). Entretanto, os campos numéricos não devem ser incluídos entre aspas!
#### Exemplo 2.3

Por favor, veja os exemplos abaixo


In [ ]:
# SELECT CustomerID e Name dos Customers do Brazil
str_query = "SELECT CustomerID, FirstName, LastName, Country FROM Customer WHERE Country = 'Brazil'" # Observe as aspas únicas
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
#SELECT InvoiceID de todas as faturas que possuem um valor total maior que 10
str_query = "SELECT InvoiceID, Total FROM Invoice  WHERE Total >= 10" # Sem aspas, já que é um valor numérico
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
#SELECT InvoiceID de todas as faturas com valor total entre 3 e 10
str_query = "SELECT InvoiceID, Total FROM Invoice  WHERE Total BETWEEN 3 AND 10"
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
#SELECT todos os clientes com nome começando com a letra F
str_query = "SELECT CustomerID, FirstName FROM Customer  WHERE FirstName LIKE 'F%'"
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
#SELECT todos os clientes que moram ou em Paris ou em Londres
str_query = "SELECT * FROM Customer WHERE City IN ('Paris','London');"
df = pd.read_sql_query(str_query, engine)
print(df.head())

**Dica**: Você pode utilizar outro SELECT dentro da cláusula WHERE!!!

```mysql
SELECT column_name(s)
FROM table_name
WHERE column_name IN (SELECT STATEMENT);
```

#### 2.8.4. Operadores AND, OR e NOT
A cláusula ```WHERE``` pode ser combinada com os operadores ```AND```, ```OR``` e ```NOT```.

Os operadores AND e OR são utilizados para filtrar registros com base em mais de uma condição:

- O operador AND exibe um registro se todas as condições separadas por AND forem VERDADEIRAS.
- O operador OR exibe um registro se qualquer uma das condições separadas por OR for VERDADEIRA.
- O operador NOT exibe um registro se a(s) condição(ões) NÃO for(em) VERDADEIRA(s).

#### Exemplo 2.4

In [ ]:
# seleciona todos os campos da tabela "Customer" no país "Germany" e cidade de "Berlin"
str_query = "SELECT * FROM Customer WHERE Country = 'Germany' AND City = 'Berlin'"
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
# seleciona todos os campos da tabela "Customer" em que a cidade é ou "Berlin" ou "Frankfurt" no país "Germany"
str_query = "SELECT * FROM Customer WHERE Country = 'Germany' AND (City = 'Berlin' OR City = 'Frankfurt')"
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
# seleciona todos os campos da tabela "Customer" em que o nome do país inicia com 'C' mas não é o 'Canada'
str_query = "SELECT * FROM Customer WHERE Country LIKE 'C%' AND NOT Country = 'Canada'"
df = pd.read_sql_query(str_query, engine)
print(df.head())

#### 2.8.5. Valores NULL

Um campo com um valor NULL é um campo sem valor. Se um campo em uma tabela for opcional, é possível inserir um novo registro ou atualizar um registro sem adicionar um valor a este campo. Então, o campo será salvo com um valor NULL.

> **Nota**: Um valor NULL é diferente de um valor zero ou de um campo que contenha espaços. Um campo com um valor NULL é aquele que foi deixado em branco durante a criação do registro!

Então, como testar os valores NULL? Bem, não é possível testar para valores NULL com operadores de comparação, tais como =, <, ou <>. Então, para fazer isso, teremos que utilizar os operadores ```IS NULL``` e ```IS NOT NULL``` em seu lugar:

#### Exemplo 2.5

In [ ]:
#Seleciona todos os Clientes que não forneceram um número de Fax, mas que forneceram um número de telefone
str_query = "SELECT CustomerID, Phone, Fax FROM Customer WHERE FAX IS NULL AND Phone IS NOT NULL"
df = pd.read_sql_query(str_query, engine)
print(df)

#### 2.8.6. Joins

Uma cláusula de [```JOIN```](https://www.w3schools.com/sql/sql_join.asp) é utilizada para combinar linhas de duas ou mais tabelas, com base em uma ou mais colunas relacionadas entre elas.

Vejamos as tabelas ```Customer``` e ```Invoice```...

In [ ]:
#SELECT Customer
str_query = "SELECT CustomerID, FirstName, LastName FROM Customer"
df = pd.read_sql_query(str_query, engine)
print(df.head())

In [ ]:
#SELECT Invoice
str_query = "SELECT InvoiceID, CustomerID, InvoiceDate, Total FROM Invoice"
df = pd.read_sql_query(str_query, engine)
print(df.head())

Observe que a coluna ```CustomerID``` na tabela ```Invoice``` se refere ao ```CustomerID``` na tabela ```Customer```. A relação entre as duas tabelas acima é então a coluna  ```CustomerID``` !

Portanto, agora que detectamos uma relação entre as tabelas, há muitas maneiras de juntar ("JOIN") estas duas tabelas:

- **(INNER) JOIN**: Retorna registros que têm valores correspondentes em ambas as tabelas
- **LEFT (OUTER) JOIN**: Retorna todos os registros da tabela da esquerda, e os registros combinados da tabela da direita
- **RIGHT (OUTER) JOIN**:  Retorna todos os registros da tabela da direita, e os registros combinados da tabela da esquerda
- **FULL (OUTER) JOIN**: Retorna todos os registros quando houver uma correspondência na tabela da esquerda ou da direita

![Imgur](https://i.imgur.com/aR5don0.png)

Logo, a sintaxe para juntar as duas tabelas é:

```mysql
SELECT column_name(s)
FROM table1
INNER JOIN table2
ON table1.column_name = table2.column_name;
```

```mysql
SELECT column_name(s)
FROM table1
LEFT JOIN table2
ON table1.column_name = table2.column_name;
```

```mysql
SELECT column_name(s)
FROM table1
RIGHT JOIN table2
ON table1.column_name = table2.column_name;
```

```mysql
SELECT column_name(s)
FROM table1
FULL OUTER JOIN table2
ON table1.column_name = table2.column_name;
```

#### Exemplo 2.6

Vamos fazer um INNER JOIN usando as tabelas ```Customer``` e ```Invoice```!

In [ ]:
# INNER JOIN
str_query = """SELECT t1.CustomerID, t1.FirstName, t1.LastName, t2.InvoiceID, t2.InvoiceDate
               FROM Customer as t1
               INNER JOIN Invoice as t2
                   ON t1.CustomerID = t2.CustomerID"""
df = pd.read_sql_query(str_query, engine)
print(df)

**Dica**: Os apelidos no SQL são usados para dar um nome temporário a uma tabela ou a uma coluna em uma tabela. Os apelidos são frequentemente usados para tornar os nomes das colunas mais legíveis. Um apelido só existe para a duração da consulta:

```mysql
SELECT column_name
FROM table_name AS alias_name;
```

#### 2.8.7. Funções agregadoras

O SQL tem algumas funções de agregação padrões:

- ```MIN```: retorna o menor valor da coluna selecionada
- ```MAX```: retorna o maior valor da coluna selecionada.
- ```COUNT```: retorna o número de linhas que correspondem a um critério específico.
- ```AVG```: retorna o valor médio de uma coluna numérica
- ```SUM```: retorna a soma total de uma coluna numérica

#### Exemplo 2.7

Vamos agregar a coluna ```Milliseconds``` para analisar a tabela ```Track```:

In [ ]:
# Agregando a coluna Milliseconds
str_query = """SELECT MIN(Milliseconds)/1000 as min_duration,
                      MAX(Milliseconds)/1000 as max_duration,
                      COUNT(Milliseconds)as number_tracks,
                      AVG(Milliseconds)/1000 as avg_track_length,
                      SUM(Milliseconds)/1000 as total_track_length
               FROM Track"""
df = pd.read_sql_query(str_query, engine)
print(df)

#### 2.8.8. GROUP BY

A declaração GROUP BY é frequentemente utilizada com funções agregadas (COUNT, MAX, MIN, SUM, AVG) para agrupar os resultados definidos por uma ou mais colunas. A sua sintaxe é:

```mysql
SELECT column_name(s)
FROM table_name
WHERE condition
GROUP BY column_name(s);
```


#### Exemplo 2.8

In [ ]:
# HÁ QUANTOS CLIENTES POR PAÍS?
str_query = """SELECT Country, COUNT(CustomerID) as number_customers
               FROM Customer
               GROUP BY Country
               ORDER BY number_customers desc"""
df = pd.read_sql_query(str_query, engine)
print(df)

In [ ]:
# QUAL O TRACK MAIS CARO PARA CADA GÊNERO?
str_query = """SELECT t2.Name as Genre, MAX(t1.UnitPrice) as max_price
               FROM Track as t1
               INNER JOIN Genre as t2
                   ON t1.GenreID = t2.GenreID
               GROUP BY Genre
               ORDER BY max_price desc"""
df = pd.read_sql_query(str_query, engine)
print(df)

<a id="pan"></a>
## Fim da segunda parte da aula!
Com a conclusão da segunda parte da aula, esperamos que você tenha aumentado seu repertório de técnicas para extração de dados, tomando conhecimento das principais variações de fontes e formatos.
Certifique-se de tirar todas suas dúvidas conosco!

Até a próxima parte!!